In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from datetime import date
import math
from tqdm import tqdm
from collections import defaultdict, Counter
import psutil
import shutil
import gc
from lightgbm import LGBMRanker, log_evaluation, early_stopping
tqdm.pandas()

# ============================================================
# PANDAS PODESAVANJA
# ============================================================
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 20)

# ============================================================
# UCITAVANJE TABELA
# ============================================================
policy_data = pd.read_csv('../data/data_from_db_as_csv/new_polisa_202603141519.csv', dtype={
    'ponuda_id': 'Int32',
    'ugo_jmbg': str
})

ins_cli_data = pd.read_csv('../data/data_from_db_as_csv/__ins_klijenti__202603141320.csv', dtype={
    'mat_br': str,
    'klijent_id': 'Int32'
})

np_cli_data = pd.read_csv('../data/data_from_db_as_csv/new_polisa_klijent_202603141515.csv', dtype={
    'osig_jmbg': str,
    'klijent_id': 'Int32',
    'ponuda_id': 'Int32'
})

cli_roles = pd.read_csv('../data/data_from_db_as_csv/new_polisa_uloga_202603172039.csv')

# ============================================================
# POMOCNE FUNKCIJE
# ============================================================
def print_duplicates(df, subset):
    print("\n")
    print("############ DUPLICATES CHECK ##############")
    print("############ EXPLOSION CHECK ##############")
    print(subset)
    df = df.copy()
    print("Broji koliko ukupno ima duplikata:", df.duplicated(subset=subset).sum())
    print("Broji koliko ima grupa sa duplikatima:", df.groupby(subset).size().gt(1).sum())
    print("Ukupno redova koji pripadaju duplikat grupama:", df[df.duplicated(subset=subset, keep=False)].shape[0])
    print("############ END ##############")
    print("\n")

process = psutil.Process(os.getpid())
def get_ram_mb():
    return process.memory_info().rss / 1024 / 1024

# ============================================================
# CISCENJE POLICY DATA
# ============================================================
policy_data = policy_data[(policy_data['polisa_id'].notna() & policy_data['polisa_id'] != 0)]
policy_data = policy_data.drop_duplicates('polisa_id', keep='first')

policy_data['dat_izdavanja'] = pd.to_datetime(policy_data['dat_izdavanja'])
policy_data['godina'] = policy_data['dat_izdavanja'].dt.year
policy_data['days_old'] = (pd.Timestamp.today() - policy_data['dat_izdavanja']).dt.days
policy_data['years_old'] = policy_data['days_old'] / 365.25

# ============================================================
# GATHERING CLI INFO
# ============================================================
def gather_cli_info_fastest(df):
    df = df.sort_values('klijent_id')
    df_filled = df.groupby('klijent_id').transform('bfill')
    df_filled['klijent_id'] = df['klijent_id'].values
    return df_filled.drop_duplicates('klijent_id', keep='first')

def extract_serial_num_Data(serial_number):
    if pd.isna(serial_number) or serial_number == '' or len(serial_number) != 13:
        return np.nan, np.nan, np.nan
    sn = str(serial_number).zfill(13)
    year = int(sn[4:7])
    year = year + 2000 if year < 800 else 1000 + year
    age = date.today().year - year
    birth_place = sn[7:9]
    gender = 1 if int(sn[9:12]) < 500 else 0
    return age, birth_place, gender

def clear_data(df):
    df = df.copy()
    df['is_foreign'] = df['br_pasosa_y'].apply(lambda x: 1 if pd.notna(x) and x != '' else 0)
    df['ind_rezident'] = df['ind_rezident'].apply(lambda x: 1 if x == 'D' else 0)
    df['ind_stranac'] = df['ind_stranac'].apply(lambda x: 1 if x == 'D' else 0)
    df['is_foreign'] = df['is_foreign'].fillna(df['ind_rezident']).fillna(df['ind_stranac'])
    df.drop(['ind_rezident', 'ind_stranac', 'drzava_id_x', 'drzava_id_y', 'br_pasosa_x', 'br_pasosa_y'], axis=1, inplace=True)
    df['ownerType'] = df['sif_svojina'].fillna(df['osig_svojina'])
    df.loc[df['ownerType'] == 2, 'ownerType'] = 0
    df.drop(['sif_svojina', 'osig_svojina'], axis=1, inplace=True)
    df[['age', 'birth_place', 'gender']] = (
        df.loc[(df['ownerType'] == 1) & (df['is_foreign'] == 0), 'mat_br']
        .apply(extract_serial_num_Data)
        .apply(pd.Series)
    )
    df['datum_rodjenja'] = pd.to_datetime(df['datum_rodjenja_x'].fillna(df['datum_rodjenja_y']), errors='coerce')
    df['years'] = date.today().year - df['datum_rodjenja'].dt.year
    df['age'] = df['age'].fillna(df['years'])
    df.drop(['datum_rodjenja_x', 'datum_rodjenja_y', 'datum_rodjenja', 'years'], axis=1, inplace=True)
    df.loc[df['pol_id_x'] == 2, 'pol_id_x'] = 0
    df.loc[df['pol_id_y'] == 2, 'pol_id_y'] = 0
    df['gender'] = df['gender'].fillna(df['pol_id_x']).fillna(df['pol_id_y'])
    df.drop(['pol_id_x', 'pol_id_y'], axis=1, inplace=True)
    df['merrige_status'] = df['sif_bracni_status_x'].fillna(df['sif_bracni_status_y'])
    df.loc[df['merrige_status'] == 2, 'merrige_status'] = 0
    df.drop(['sif_bracni_status_x', 'sif_bracni_status_y'], axis=1, inplace=True)
    df['unknown_job'] = df['nedef_delatnost_y'].fillna(df['nedef_delatnost_x'])
    df.drop(['nedef_delatnost_y', 'nedef_delatnost_x'], axis=1, inplace=True)
    df['job_type'] = df['sif_delatnost_y'].fillna(df['sif_delatnost_x'])
    df.drop(['sif_delatnost_y', 'sif_delatnost_x'], axis=1, inplace=True)
    df['driving_licence_date'] = df['dat_vozacke_y'].fillna(df['dat_vozacke_x'])
    df.drop(['dat_vozacke_y', 'dat_vozacke_x'], axis=1, inplace=True)
    text_cols = ['mat_br', 'naziv', 'naziv1', 'king_id_x', 'mesto_rodj_x', 'osig_jmbg', 'osig_naziv', 'osig_naziv1',
                 'osig_ulica', 'osig_mesto', 'osig_telefon2', 'ind_info_ponuda', 'king_id_y', 'osig_telefon1', 'osig_mail',
                 'mesto_rodj_y', 'procenat', 'sif_mj']
    df.drop(text_cols, axis=1, inplace=True)
    return df

def feature_engineering(df):
    df = df.copy()
    df['age_bucket'] = pd.cut(df['age'], bins=[0, 25, 35, 50, 65, 100], labels=[0, 1, 2, 3, 4])
    df['driving_licence_date'] = pd.to_datetime(df['driving_licence_date'], errors='coerce')
    today_year = pd.Timestamp.today().year
    df['driving_experience'] = today_year - df['driving_licence_date'].dt.year
    df['driving_experience'] = df['driving_experience'].astype(float)
    df['driving_experience'] = df['driving_experience'].fillna(
        df.groupby('age_bucket', observed=True)['driving_experience'].transform('median')
    )
    df['driving_experience'] = df['driving_experience'].fillna(df['driving_experience'].median())
    df['merrige_status'] = df['merrige_status'].astype(float)
    df['merrige_status'] = df['merrige_status'].fillna(
        df.groupby('age_bucket', observed=True)['merrige_status'].transform('median')
    )
    df['merrige_status'] = df['merrige_status'].fillna(df['merrige_status'].median())
    df = df.drop('driving_licence_date', axis=1)
    return df

# ============================================================
# MERGE I CISCENJE KLIJENATA
# ============================================================
cli_data = ins_cli_data.merge(np_cli_data, on="klijent_id", how="inner")
cli_data = gather_cli_info_fastest(cli_data)
cli_data = clear_data(cli_data)

corrupted_data_ids = cli_data[(cli_data['age'] < 18) | (cli_data['age'] > 90)]['klijent_id'].values
cli_data.loc[cli_data['klijent_id'].isin(corrupted_data_ids), 'age'] = \
    cli_data.groupby(['merrige_status', 'osig_opstina'])['age'].transform('median')

cli_data = cli_data.drop(['unknown_job', 'job_type'], axis=1)
cli_data = feature_engineering(cli_data)

# REASSEMBLE - merge sa np_cli_data
cli_data = cli_data.merge(np_cli_data[['klijent_id', 'ponuda_id', 'sif_uloga']], on='klijent_id', how='inner')
cli_data['ponuda_id'] = cli_data['ponuda_id_y']
cli_data = cli_data.drop(['ponuda_id_y', 'ponuda_id_x'], axis=1)
cli_data['sif_uloga'] = cli_data['sif_uloga_y']
cli_data = cli_data.drop(['sif_uloga_y', 'sif_uloga_x'], axis=1)

# FIX: brisemo prave duplikate po svim kljucnim kolonama, NE samo po klijent_id
cli_duplicates = cli_data[cli_data.duplicated(['klijent_id', 'ponuda_id', 'sif_uloga'])].shape[0]
print(f'Duplicated clients: {cli_duplicates}')
if cli_duplicates > 0:
    cli_data = cli_data.drop_duplicates(['klijent_id', 'ponuda_id', 'sif_uloga'], keep='first')

# MERGE SA POLISAMA
policy_client_data = policy_data.merge(cli_data, on='ponuda_id', how='inner')

# FILTRIRANJE NA UGOVARACA
cli_roles = cli_roles[cli_roles['opis'].str.contains('Ugovarač', na=False)]
policy_client_data = policy_client_data[policy_client_data['sif_uloga'].isin(cli_roles['sif_uloga'].values)]

def clean_policy_client_data(df):
    df = df.copy()
    cols_to_delete = ['sif_preuzmi', 'preuzmi_no', 'preuzmi_id', 'ugo_jmbg', 'ugo_svojina', 'ugo_naziv', 'ugo_naziv1',
                      'ugo_ulica', 'ugo_kuc_br', 'ugo_mesto', 'ugo_posta', 'ugo_kanton', 'ugo_opstina', 'ugo_mesto_id',
                      'ugo_telefon1', 'ugo_telefon2', 'ugo_mail', 'osig_jmbg', 'osig_svojina', 'osig_naziv', 'osig_naziv1',
                      'osig_ulica', 'osig_kuc_br_x', 'osig_mesto', 'osig_posta_x', 'osig_kanton_x', 'osig_opstina_x',
                      'osig_mesto_id_x', 'osig_telefon1', 'osig_telefon2', 'osig_mail', 'mesto_izdavanja',
                      'miro_polisa_no', 'sif_ikanton', 'sif_iopstina', 'ugo_del', 'king_id', 'ugo_br_pasosa',
                      'osig_br_pasosa', 'registarski_broj_polise', 'akcija_id']
    df = df.drop(cols_to_delete, axis=1)
    return df

policy_client_data = clean_policy_client_data(policy_client_data)

# Oslobodi memoriju
ins_cli_data = None
np_cli_data = None
cli_roles = None
cli_data = None
gc.collect()

# ============================================================
# MARKOV MODEL
# ============================================================
def _build_markov_transition(df):
    trans = Counter()
    for _, g in df.groupby('klijent_id'):
        g = g.sort_values('dat_izdavanja')
        types = g['sif_vrsta'].tolist()
        for i in range(len(types) - 1):
            if types[i] != types[i+1]:
                trans[(types[i], types[i+1])] += 1
    out = pd.DataFrame(
        [(a, b, c) for (a, b), c in trans.items()],
        columns=['from_type', 'to_type', 'cnt']
    )
    out['prob'] = out.groupby('from_type')['cnt'].transform(lambda x: x / x.sum())
    return out

# ============================================================
# FUTURE PAIRS
# ============================================================
def _build_future_pairs(df):
    df = df.sort_values(['klijent_id', 'dat_izdavanja', 'ponuda_id']).copy()
    rows = []
    for _, g in df.groupby('klijent_id'):
        g = g.sort_values(['dat_izdavanja', 'ponuda_id'])
        ponuda_ids = g['ponuda_id'].tolist()
        types = g['sif_vrsta'].tolist()
        for i in range(len(types) - 1):
            if types[i] == types[i + 1]:
                continue
            rows.append((ponuda_ids[i], types[i + 1]))
    return (
        pd.DataFrame(rows, columns=['ponuda_id', 'candidate_type'])
        .drop_duplicates()
        .assign(label=1)
    )

# ============================================================
# CLIENT HISTORY MATRIX
# FIX: prima df koji je vec train ili val, ne cijeli dataset
# ============================================================
def _build_client_history_matrix(df):
    df = df.sort_values(['klijent_id', 'dat_izdavanja']).copy()
    dummies = pd.get_dummies(df['sif_vrsta'], prefix='type')
    hist = pd.concat(
        [df[['klijent_id', 'dat_izdavanja', 'ponuda_id']].reset_index(drop=True),
         dummies.reset_index(drop=True)],
        axis=1
    )
    feature_cols = dummies.columns
    hist[feature_cols] = (
        hist.groupby(hist['klijent_id'])[feature_cols]
        .cumsum()
        .groupby(hist['klijent_id'])
        .shift(1)
        .fillna(0)
    )
    return hist.drop(columns=['klijent_id', 'dat_izdavanja'])

# ============================================================
# FEATURE DEFINICIJE
# FIX: features lista je sada funkcija da izbjegnemo mutaciju pri
# visestrukim pozivima pipeline-a
# ============================================================
BASE_FEATURES = [
    'label', 'klijent_id', 'ponuda_id', 'sif_vrsta',
    'n_policies_before', 'had_type_before', 'dat_izdavanja', 'days_since_last_policy',
    'candidate_type', 'cnt_type_before',
    'markov_prob',
    'avg_premium_past', 'avg_insurance_sum_past', 'age', 'gender', 'ownerType', 'merrige_status', 'is_foreign',
    'birth_place', 'osig_mesto_id_y', 'osig_opstina_y',
    'osig_kanton_y', 'osig_posta_y', 'sif_uloga', 'age_bucket'
]

TRAINING_FEATURES = [
    'n_policies_before', 'had_type_before', 'candidate_type', 'days_since_last_policy', 'cnt_type_before',
    'markov_prob',
    'avg_premium_past', 'avg_insurance_sum_past', 'age', 'gender', 'ownerType', 'merrige_status', 'is_foreign',
    'birth_place', 'osig_mesto_id_y', 'osig_opstina_y',
    'osig_kanton_y', 'osig_posta_y', 'sif_uloga', 'age_bucket'
]

def _get_categorical_training_features():
    return [
        'had_type_before', 'candidate_type', 'gender', 'ownerType', 'merrige_status', 'is_foreign',
        'birth_place', 'osig_mesto_id_y', 'osig_opstina_y', 'osig_kanton_y', 'osig_posta_y', 'sif_uloga', 'age_bucket'
    ]

# ============================================================
# CHUNK BUILDER
# ============================================================
def _build_training_chunks(df, future_pairs, mt_dict, candidates, client_history, features, chunk_size=5000, chunks_dir='chunks_temp'):
    df = df.copy()
    os.makedirs(chunks_dir, exist_ok=True)
    n_chunks = (len(df) // chunk_size) + 1
    saved_chunks = 0

    for i, start in enumerate(tqdm(range(0, len(df), chunk_size), total=n_chunks, desc="Chunks", colour="green")):
        chunk = df.iloc[start:start + chunk_size].copy()

        # CROSS JOIN
        chunk_dataset = chunk.assign(key=1).merge(
            candidates.assign(key=1), on='key'
        ).drop('key', axis=1)

        # MARKOV
        chunk_dataset['markov_prob'] = chunk_dataset.set_index(
            ['sif_vrsta', 'candidate_type']
        ).index.map(mt_dict)
        chunk_dataset['markov_prob'] = chunk_dataset['markov_prob'].fillna(0)

        # CLIENT HISTORY
        chunk_dataset = chunk_dataset.merge(client_history, on=['ponuda_id'], how='inner')
        history_cols = [c for c in client_history.columns if c != 'ponuda_id']
        chunk_dataset[history_cols] = chunk_dataset[history_cols].fillna(0)

        # LABEL MERGE
        future_pairs_chunk = future_pairs[future_pairs['ponuda_id'].isin(chunk['ponuda_id'])]
        chunk_dataset = chunk_dataset.merge(
            future_pairs_chunk,
            on=['ponuda_id', 'candidate_type'],
            how='left'
        )
        chunk_dataset['label'] = chunk_dataset['label'].fillna(0).astype(int)

        chunk_dataset[features].to_pickle(f'{chunks_dir}/chunk_{i}.pkl')
        saved_chunks += 1

        del chunk, chunk_dataset, future_pairs_chunk
        gc.collect()

    return saved_chunks, chunks_dir

def merge_chunks(saved_chunks, chunks_dir):
    dataset = pd.concat([
        pd.read_pickle(f'{chunks_dir}/chunk_{i}.pkl')
        for i in range(saved_chunks)
    ], ignore_index=True)
    print(f"   Final dataset shape: {dataset.shape}")
    shutil.rmtree(chunks_dir)
    print("✅ Done!")
    return dataset

# ============================================================
# GLAVNI PIPELINE
# FIX lista gresaka:
#   1. client_history se gradi ODVOJENO za train i val (ne na cijelom df)
#   2. features lista je BASE_FEATURES kopija - nema mutacije
#   3. cap_candidates se primjenjuje i na val_dataset
#   4. hit_rate_at_k koristi np.argsort(-preds) svuda
# ============================================================
df = policy_client_data.copy()

# Samo klijenti sa vise od jedne polise
df = df.groupby(['klijent_id']).filter(lambda x: len(x) > 1)
print(f'Ukupni broj polisa sa klijentima koji imaju vise od jedne polise: {len(df)}')
print(f"Broj jedinstvenih klijenata: {df['klijent_id'].nunique()}")

# 1. CLEANING & SORTING
print("✅ [1/7] Cleaning & sorting...")
df['dat_izdavanja'] = pd.to_datetime(df['dat_izdavanja'])
df = df.sort_values(['klijent_id', 'dat_izdavanja']).reset_index(drop=True)

# 2. HISTORY FEATURES
print("✅ [2/7] Building history features...")
df['days_since_last_policy'] = df.groupby('klijent_id')['dat_izdavanja'].diff().dt.days
df['n_policies_before'] = df.groupby('klijent_id').cumcount()
df['cnt_type_before'] = df.groupby(['klijent_id', 'sif_vrsta']).cumcount()
df['had_type_before'] = (df['cnt_type_before'] > 0).astype(int)

g = df.groupby('klijent_id')
df['avg_premium_past'] = g['premija_ukupno'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df['avg_insurance_sum_past'] = g['suma_osiguranja'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())

# VREMENSKI SPLIT - mora biti PRIJE gradnje historije i markov matrice
train_df = df[df['dat_izdavanja'] < '2025-12-01'].copy()
val_df   = df[df['dat_izdavanja'] >= '2025-12-01'].copy()

# 3. MARKOV - samo na train setu
print("✅ [3/7] Building Markov transition matrix...")
markov_transitions = _build_markov_transition(train_df)
mt_dict = markov_transitions.set_index(['from_type', 'to_type'])['prob'].to_dict()
print(f"   Co-occurrence shape: {markov_transitions.shape}")

# Provjeri coverage - tipovi u val koji nisu u train
all_types_train = set(train_df['sif_vrsta'].unique())
all_types_val   = set(val_df['sif_vrsta'].unique())
unseen_types = all_types_val - all_types_train
if unseen_types:
    print(f"⚠️  Tipovi polisa u val koji nisu u train (markov_prob=0): {unseen_types}")

# 4. FUTURE PAIRS
print("✅ [4/7] Building future pairs (labels)...")
future_pairs_train = _build_future_pairs(train_df)
future_pairs_val   = _build_future_pairs(val_df)
print(f"   Future pairs train shape: {future_pairs_train.shape}")
print(f"   Future pairs val shape: {future_pairs_val.shape}")

# 5. KANDIDATI - svi tipovi iz cijelog df-a
all_types = df['sif_vrsta'].unique()
candidates = pd.DataFrame({'candidate_type': all_types})
print(f"   Unique policy types (candidates): {len(all_types)}")

# 6. CLIENT HISTORY - FIX: odvojeno za train i val!
print("✅ [5/7] Building candidate history lookup...")
client_history_train = _build_client_history_matrix(train_df)
client_history_val   = _build_client_history_matrix(val_df)
print(f"   Train history shape: {client_history_train.shape}")
print(f"   Val history shape:   {client_history_val.shape}")

# FIX: features lista - koristimo kopiju BASE_FEATURES, dodajemo type_ kolone
history_type_cols = [c for c in client_history_train.columns if c != 'ponuda_id']
features = BASE_FEATURES + history_type_cols

# 7. UKLANJANJE ZADNJE POLISE
train_df = train_df.sort_values(['klijent_id', 'dat_izdavanja'])
val_df   = val_df.sort_values(['klijent_id', 'dat_izdavanja'])
train_df = train_df[train_df.groupby('klijent_id').cumcount(ascending=False) != 0]
val_df   = val_df[val_df.groupby('klijent_id').cumcount(ascending=False) != 0]

print(f'Train polisa nakon uklanjanja zadnje: {len(train_df)}')
print(f'Val polisa nakon uklanjanja zadnje: {len(val_df)}')

gc.collect()

# 8. CHUNK PROCESSING
print("✅ [6/7] Processing chunks...")
saved_chunks_train, train_dir = _build_training_chunks(
    train_df, future_pairs_train, mt_dict, candidates, client_history_train, features,
    chunk_size=5000, chunks_dir='chunks_temp_train'
)
saved_chunks_val, val_dir = _build_training_chunks(
    val_df, future_pairs_val, mt_dict, candidates, client_history_val, features,
    chunk_size=5000, chunks_dir='chunks_temp_val'
)

# 9. KONKATENACIJA
print("✅ [7/7] Concatenating chunks...")
train_dataset = merge_chunks(saved_chunks_train, train_dir)
val_dataset   = merge_chunks(saved_chunks_val, val_dir)

# CATEGORICAL FEATURES
cat_cols = _get_categorical_training_features()
for col in cat_cols:
    train_dataset[col] = train_dataset[col].astype('category')
    val_dataset[col]   = val_dataset[col].astype('category')

# ============================================================
# PRIPREMA ZA MODEL
# ============================================================

# FIX: cap_candidates i za val_dataset
def cap_candidates(df, max_per_group):
    return (
        df
        .groupby(['klijent_id', 'dat_izdavanja'], group_keys=False)
        .head(max_per_group)
    )

max_per_group = len(all_types) * 7
train_dataset = cap_candidates(train_dataset, max_per_group)
val_dataset   = cap_candidates(val_dataset, max_per_group)  # FIX: nedostajalo

# SORT - obavezno za ranking
train_dataset = train_dataset.sort_values(['klijent_id', 'dat_izdavanja']).reset_index(drop=True)
val_dataset   = val_dataset.sort_values(['klijent_id', 'dat_izdavanja']).reset_index(drop=True)

# Azurirani training_features sa type_ kolonama
training_features = TRAINING_FEATURES + history_type_cols

X_train = train_dataset[training_features].copy()
for col in _get_categorical_training_features():
    X_train[col] = X_train[col].astype('category').cat.codes
y_train = train_dataset['label'].astype(int)

X_val = val_dataset[training_features].copy()
for col in _get_categorical_training_features():
    X_val[col] = X_val[col].astype('category').cat.codes
y_val = val_dataset['label'].astype(int)

# GROUP
group_train = train_dataset.groupby(['klijent_id', 'dat_izdavanja']).size().to_numpy()
group_val   = val_dataset.groupby(['klijent_id', 'dat_izdavanja']).size().to_numpy()

# SANITY CHECK
assert sum(group_train) == len(X_train), "❌ group_train i X_train nisu uskladjeni!"
assert sum(group_val) == len(X_val),     "❌ group_val i X_val nisu uskladjeni!"

print("✅ Train rows:", len(X_train), "| Groups:", len(group_train))
print("✅ Val rows:",   len(X_val),   "| Groups:", len(group_val))

# ============================================================
# MODEL
# ============================================================
model = LGBMRanker(
    objective='lambdarank',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31
)

model.fit(
    X_train,
    y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    eval_at=[1, 3, 5],
    callbacks=[
        early_stopping(50),
        log_evaluation(10)
    ]
)

# ============================================================
# EVALUACIJA
# FIX: sve funkcije koriste np.argsort(-preds) za TOP-k
# ============================================================
def hit_rate_at_k(model, X, y, group, k=1):
    """
    Hit Rate @k: Da li se relevantan item nalazi medju top-k predikcija.
    FIX: np.argsort(-preds)[:k] - uzima NAJVISE score-ove.
    """
    start = 0
    hits = 0
    total = 0
    preds_all = model.predict(X)

    for g in group:
        end = start + g
        preds = preds_all[start:end]
        labels = y.iloc[start:end].values

        # FIX: - ispred preds = silazni sort, [:k] = prvih k
        top_k_idx = np.argsort(-preds)[:k]

        if np.any(labels[top_k_idx] == 1):
            hits += 1

        total += 1
        start = end

    return hits / total if total > 0 else 0.0

def mrr_at_k(model, X, y, group, k=5):
    """
    Mean Reciprocal Rank @k
    FIX: np.argsort(preds)[::-1] -> isti efekt kao argsort(-preds)
    """
    start = 0
    reciprocal_ranks = []
    preds_all = model.predict(X)

    for g in group:
        end = start + g
        preds = preds_all[start:end]
        labels = y.iloc[start:end].values

        sorted_idx = np.argsort(-preds)  # FIX: silazno

        found = False
        for rank, idx in enumerate(sorted_idx[:k], 1):
            if labels[idx] == 1:
                reciprocal_ranks.append(1.0 / rank)
                found = True
                break

        if not found:
            reciprocal_ranks.append(0.0)

        start = end

    return np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0

def ndcg_at_k(model, X, y, group, k=3):
    start = 0
    total_ndcg = 0
    n_groups = 0
    preds_all = model.predict(X)

    for g in group:
        end = start + g
        preds = preds_all[start:end]
        labels = y.iloc[start:end].values

        order = np.argsort(-preds)  # FIX: silazno
        labels_sorted = labels[order][:k]
        dcg = np.sum(labels_sorted / np.log2(np.arange(2, k+2)))

        ideal = np.sort(labels)[::-1][:k]
        idcg = np.sum(ideal / np.log2(np.arange(2, k+2)))

        if idcg > 0:
            total_ndcg += dcg / idcg
            n_groups += 1

        start = end

    return total_ndcg / n_groups if n_groups > 0 else 0.0

# REZULTATI
print("\n" + "=" * 50)
print("EVALUACIJA NA VAL SETU")
print("=" * 50)
for k in [1, 3, 5]:
    hr  = hit_rate_at_k(model, X_val, y_val, group_val, k=k)
    mrr = mrr_at_k(model, X_val, y_val, group_val, k=k)
    ndcg = ndcg_at_k(model, X_val, y_val, group_val, k=k)
    print(f"HR@{k}: {hr:.4f} | MRR@{k}: {mrr:.4f} | NDCG@{k}: {ndcg:.4f}")

print("\n" + "=" * 50)
print("OVERFITTING CHECK (train vs val)")
print("=" * 50)
for k in [1, 3]:
    train_hr = hit_rate_at_k(model, X_train, y_train, group_train, k=k)
    val_hr   = hit_rate_at_k(model, X_val, y_val, group_val, k=k)
    print(f"HR@{k} -> Train: {train_hr:.4f} | Val: {val_hr:.4f} | Gap: {train_hr - val_hr:.4f}")

# FEATURE IMPORTANCE
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)
print("\nTop 10 feature importances:")
print(fi.head(10).to_string(index=False))

C:\Users\Korisnik\AppData\Local\Temp\ipykernel_14520\3961603493.py:26: DtypeWarning: Columns (61,65,80,82) have mixed types. Specify dtype option on import or set low_memory=False.
  policy_data = pd.read_csv('../data/data_from_db_as_csv/new_polisa_202603141519.csv', dtype={
C:\Users\Korisnik\AppData\Local\Temp\ipykernel_14520\3961603493.py:36: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  np_cli_data = pd.read_csv('../data/data_from_db_as_csv/new_polisa_klijent_202603141515.csv', dtype={


Duplicated clients: 54
Ukupni broj polisa sa klijentima koji imaju vise od jedne polise: 15148
Broj jedinstvenih klijenata: 4418
✅ [1/7] Cleaning & sorting...
✅ [2/7] Building history features...
✅ [3/7] Building Markov transition matrix...
   Co-occurrence shape: (105, 4)
✅ [4/7] Building future pairs (labels)...
   Future pairs train shape: (2928, 3)
   Future pairs val shape: (223, 3)
   Unique policy types (candidates): 14
✅ [5/7] Building candidate history lookup...
   Train history shape: (13314, 15)
   Val history shape:   (1834, 15)
Train polisa nakon uklanjanja zadnje: 9062
Val polisa nakon uklanjanja zadnje: 708
✅ [6/7] Processing chunks...


Chunks: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.04it/s]


✅ [7/7] Concatenating chunks...
   Final dataset shape: (126868, 39)
✅ Done!
   Final dataset shape: (9912, 39)
✅ Done!
✅ Train rows: 124460 | Groups: 7157
✅ Val rows: 9744 | Groups: 512
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006508 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2547
[LightGBM] [Info] Number of data points in the train set: 124460, number of used features: 33
Training until validation scores don't improve for 50 rounds
[10]	valid_0's ndcg@1: 0.783203	valid_0's ndcg@3: 0.857413	valid_0's ndcg@5: 0.880388
[20]	valid_0's ndcg@1: 0.789062	valid_0's ndcg@3: 0.866602	valid_0's ndcg@5: 0.885884
[30]	valid_0's ndcg@1: 0.802734	valid_0's ndcg@3: 0.873967	valid_0's ndcg@5: 0.890401
[40]	valid_0's ndcg@1: 0.798828	valid_0's ndcg@3: 0.870713	valid_0's ndcg@5: 0.888382
[50]	valid_0's ndcg@1: 0.802734	valid_0's ndcg@3: 0.86

In [2]:
import numpy as np
import pandas as pd

def add_sequence_recommendation_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds sequence-aware and affinity features for next-policy recommendation.
    Assumes df is sorted by ['klijent_id', 'dat_izdavanja'].
    """

    df = df.copy()

    # ============================================================
    # 1. BASIC SEQUENCE (LAST-K)
    # ============================================================
    g = df.groupby('klijent_id')

    df['last_type_1'] = g['sif_vrsta'].shift(1)
    df['last_type_2'] = g['sif_vrsta'].shift(2)
    df['last_type_3'] = g['sif_vrsta'].shift(3)

    # Repeat signal (very strong feature)
    df['type_repeat'] = (df['sif_vrsta'] == df['last_type_1']).astype(int)

    # ============================================================
    # 2. RECENCY FEATURES
    # ============================================================
    df['days_since_last_policy'] = g['dat_izdavanja'].diff().dt.days

    df['recency_weight'] = 1 / (1 + df['days_since_last_policy'].fillna(999))
    df['recency_exp'] = np.exp(-0.01 * df['days_since_last_policy'].fillna(999))

    # ============================================================
    # 3. CLIENT × TYPE AFFINITY
    # ============================================================
    client_type_cnt = df.groupby(['klijent_id', 'sif_vrsta']).size().reset_index(name='client_type_cnt')
    df = df.merge(client_type_cnt, on=['klijent_id', 'sif_vrsta'], how='left')

    df['n_policies_before'] = g['sif_vrsta'].cumcount()

    df['client_type_affinity'] = (
        df['client_type_cnt'] / (df['n_policies_before'] + 1)
    )

    # ============================================================
    # 4. SEQUENCE MATCH FEATURES (VERY STRONG FOR RANKING)
    # ============================================================
    df['match_last_1'] = (df['sif_vrsta'] == df['last_type_1']).astype(int)
    df['match_last_2'] = (df['sif_vrsta'] == df['last_type_2']).astype(int)

    # ============================================================
    # 5. CLEANUP (optional safety)
    # ============================================================
    df = df.replace([np.inf, -np.inf], np.nan)

    return df

df = df.sort_values(['klijent_id', 'dat_izdavanja']).reset_index(drop=True)

df = add_sequence_recommendation_features(df)

In [3]:
# Globalna popularnost tipa
type_popularity = train_df['sif_vrsta'].value_counts(normalize=True).to_dict()
df['candidate_global_popularity'] = df['candidate_type'].map(type_popularity)

# Popularnost po regiji (osig_opstina)
regional_pop = (
    train_df.groupby(['osig_opstina_y', 'sif_vrsta'])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum())
)

KeyError: 'candidate_type'

In [4]:
df['month'] = df['dat_izdavanja'].dt.month
df['quarter'] = df['dat_izdavanja'].dt.quarter

# Koliko dana od zadnje polise istog tipa
df['days_since_same_type'] = (
    df.groupby(['klijent_id', 'sif_vrsta'])['dat_izdavanja']
    .diff().dt.days
)

In [5]:
def _build_markov_with_decay(df, half_life_days=365):
    today = df['dat_izdavanja'].max()
    rows = []
    
    for _, g in df.groupby('klijent_id'):
        g = g.sort_values('dat_izdavanja')
        types = g['sif_vrsta'].tolist()
        dates = g['dat_izdavanja'].tolist()
        
        for i in range(len(types) - 1):
            if types[i] == types[i+1]:
                continue
            days_ago = (today - dates[i+1]).days
            weight = 2 ** (-days_ago / half_life_days)  # exponential decay
            rows.append((types[i], types[i+1], weight))
    
    out = pd.DataFrame(rows, columns=['from_type', 'to_type', 'weight'])
    out['prob'] = out.groupby('from_type')['weight'].transform(lambda x: x / x.sum())
    return out

In [6]:
# Segmentacija po starosti, tipu vlasništva, regiji
df['segment'] = (
    df['age_bucket'].astype(str) + '_' +
    df['ownerType'].astype(str) + '_' +
    df['osig_opstina_y'].astype(str)
)

# Za svaki segment, koja je najčešća sljedeća polisa
segment_next = (
    train_df.groupby(['segment', 'sif_vrsta', 'next_type'])  # next_type iz future_pairs
    .size()
    .reset_index(name='cnt')
)
segment_next['prob'] = segment_next.groupby(['segment', 'sif_vrsta'])['cnt'].transform(lambda x: x / x.sum())

# Ovo postaje novi feature: segment_markov_prob

KeyError: 'segment'

In [7]:
def add_renewal_features(df):
    df = df.copy()
    
    # Koliko dana je prošlo od zadnje polise ISTOG tipa
    df = df.sort_values(['klijent_id', 'sif_vrsta', 'dat_izdavanja'])
    df['days_since_same_type'] = (
        df.groupby(['klijent_id', 'sif_vrsta'])['dat_izdavanja']
        .diff().dt.days
    )
    
    # Binary: da li je ovo vjerovatno obnova (330-400 dana = godišnja obnova)
    df['likely_renewal'] = (
        (df['days_since_same_type'] >= 330) & 
        (df['days_since_same_type'] <= 400)
    ).astype(int)
    
    return df